# Benchmark Analysis

## 1. Estilo Global

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

BG        = '#1b1d22'   
FG        = '#ffffff'   
NEON      = '#ccff00'  
GRAY_BAR  = '#8d94a1'   
DIM_TEXT  = '#8d94a1'   

mpl.rcParams['font.family']     = 'DejaVu Sans'
mpl.rcParams['font.weight']     = 'bold'
mpl.rcParams['text.color']      = FG
mpl.rcParams['axes.labelcolor'] = FG
mpl.rcParams['xtick.color']     = FG
mpl.rcParams['ytick.color']     = FG
mpl.rcParams['figure.facecolor'] = BG
mpl.rcParams['axes.facecolor']   = BG

print("Melo.")


## 2. Carga de Datos

In [ ]:
ruta_carpeta = 'benchmarks'
archivos_csv = glob.glob(os.path.join(ruta_carpeta, "*.csv"))

datos_dict = {}
for archivo in archivos_csv:
    nombre_archivo = os.path.basename(archivo)
    datos_dict[nombre_archivo] = pd.read_csv(archivo)

print(f"Se cargaron {len(datos_dict)} archivos.")


## 3. Agregación de Métricas

In [ ]:
gflops_totales = {}

for nombre_archivo, df in datos_dict.items():
    if 'gflops' in df.columns:
        gflops_totales[nombre_archivo] = df['gflops'].sum()
    else:
        print(f"Advertencia: {nombre_archivo} no tiene columna 'gflops'.")

print("Total de GFLOPS por archivo:")
for nombre, total in sorted(gflops_totales.items()):
    print(f"  {nombre}: {total:.1f}")


In [ ]:
tiempos_totales = {}

for nombre_archivo, df in datos_dict.items():
    if 'tiempo_ms' in df.columns:
        tiempos_totales[nombre_archivo] = df['tiempo_ms'].sum()
    else:
        print(f"Advertencia: {nombre_archivo} no tiene columna 'tiempo_ms'.")

print("Tiempo total (ms) por archivo:")
for nombre, total in sorted(tiempos_totales.items()):
    print(f"  {nombre}: {total:.1f} ms")


## 4. GFLOPS — Comparaciones por Par de Implementaciones


In [ ]:
tamanos = ['10', '11', '12', '13', '14']

def generar_grafico_comparativo(sufijo_base, sufijo_top, etiqueta_base, etiqueta_top, titulo_grafico):
    valores_base = [gflops_totales.get(f"{t}_{sufijo_base}.csv", 0) for t in tamanos]
    valores_top  = [gflops_totales.get(f"{t}_{sufijo_top}.csv",  0) for t in tamanos]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    y = np.arange(len(tamanos))
    ancho = 0.35
    max_val = max(valores_top) if max(valores_top) > 0 else 1

    ax.barh(y + ancho / 2, valores_top,  ancho, color=NEON,     label=etiqueta_top)
    ax.barh(y - ancho / 2, valores_base, ancho, color=GRAY_BAR, label=etiqueta_base)

    for i in range(len(tamanos)):
        # Valor barra superior
        ax.text(valores_top[i]  + max_val * 0.01, y[i] + ancho / 2,
                f"{valores_top[i]:,.1f}", color=FG, va='center', fontweight='bold', fontsize=10)
        # Valor barra base
        ax.text(valores_base[i] + max_val * 0.01, y[i] - ancho / 2,
                f"{valores_base[i]:,.1f}", color=GRAY_BAR, va='center', fontweight='bold', fontsize=10)
        # Porcentaje de incremento
        if valores_base[i] > 0:
            pct = ((valores_top[i] - valores_base[i]) / valores_base[i]) * 100
            ax.text(valores_base[i] + max_val * 0.15, y[i] - ancho / 2,
                    f"↑ +{pct:.0f}%", color=NEON, va='center', fontweight='bold', fontsize=10)

    ax.set_yticks(y)
    ax.set_yticklabels([f"EXP {t}" for t in tamanos], fontweight='bold', fontsize=11)
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.title(titulo_grafico, fontsize=15, fontweight='bold', pad=20)
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2,
              frameon=True, facecolor=BG, edgecolor=GRAY_BAR, labelcolor=FG)
    plt.tight_layout()
    plt.show()

# --- Comparaciones ---
generar_grafico_comparativo('cpu',          'gpu_naive',     'CPU',        'GPU Naive',     'CPU vs GPU Naive (GFLOPS)')
generar_grafico_comparativo('gpu_naive',    'gpu_coalesced', 'GPU Naive',  'GPU Coalesced', 'GPU Naive vs GPU Coalesced (GFLOPS)')
generar_grafico_comparativo('gpu_coalesced','gpu_tiled',     'GPU Coalesced','GPU Tiled',   'GPU Coalesced vs GPU Tiled (GFLOPS)')
generar_grafico_comparativo('gpu_tiled',    'gpu_cublas',    'GPU Tiled',  'GPU CuBLAS',    'GPU Tiled vs GPU CuBLAS (GFLOPS)')
generar_grafico_comparativo('cpu',          'gpu_cublas',    'CPU',        'GPU CuBLAS',    'CPU vs GPU CuBLAS (GFLOPS)')


## 5. Tiempo Total de Ejecución — Por Exponente

In [ ]:
modos           = ['cpu', 'gpu_naive', 'gpu_coalesced', 'gpu_tiled', 'gpu_cublas']
etiquetas_modos = ['CPU', 'GPU\nNaive', 'GPU\nCoalesced', 'GPU\nTiled', 'GPU\nCuBLAS']
tamanos         = ['10', '11', '12', '13', '14']

def formatear_tiempo(ms):
    if ms <= 0:   return "0 ms"
    if ms < 1000: return f"{ms:.0f} ms"
    s = ms / 1000
    if s < 60:    return f"{s:.1f} s"
    m = s / 60
    if m < 60:    return f"{m:.1f} m"
    return f"{m / 60:.1f} h"

matriz = np.array([
    [tiempos_totales.get(f"{t}_{m}.csv", 0) for m in modos]
    for t in tamanos
], dtype=float)

log_matriz = np.log10(np.where(matriz > 0, matriz, 1e-5))
vmin, vmax = log_matriz.min(), log_matriz.max()
norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap  = mcolors.LinearSegmentedColormap.from_list(
    "bench", [BG, '#3a3f4a', GRAY_BAR, '#aacc00', NEON]
)

fig, ax = plt.subplots(figsize=(13, 6))

for row, tam in enumerate(tamanos):
    for col, modo in enumerate(modos):
        val      = matriz[row, col]
        color_bg = cmap(norm(log_matriz[row, col]))

        rect = plt.Rectangle([col, row], 1, 1, color=color_bg, zorder=1)
        ax.add_patch(rect)

        luminancia = 0.299*color_bg[0] + 0.587*color_bg[1] + 0.114*color_bg[2]
        color_txt  = BG if luminancia > 0.45 else FG

        ax.text(col + 0.5, row + 0.5, formatear_tiempo(val),
                ha='center', va='center', fontsize=11,
                fontweight='bold', color=color_txt, zorder=2)

for i in range(1, len(modos)):
    ax.axvline(i, color=BG, linewidth=2, zorder=3)
for i in range(1, len(tamanos)):
    ax.axhline(i, color=BG, linewidth=2, zorder=3)

ax.set_xlim(0, len(modos))
ax.set_ylim(0, len(tamanos))

ax.set_xticks([c + 0.5 for c in range(len(modos))])
ax.set_xticklabels(etiquetas_modos, fontsize=11, fontweight='bold', color=FG)
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

ax.set_yticks([r + 0.5 for r in range(len(tamanos))])
ax.set_yticklabels([f"EXP {t}" for t in tamanos], fontsize=11, fontweight='bold', color=FG)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("Tiempo Total de Ejecución", fontsize=22, fontweight='bold', pad=48, color=FG)

plt.tight_layout()
plt.show()

## 6. GFLOPS de Todos los Modos

In [ ]:
tamanos      = ['10', '11', '12', '13', '14']
modos        = ['cpu', 'gpu_naive', 'gpu_coalesced', 'gpu_tiled', 'gpu_cublas']
etiquetas    = ['CPU', 'Naive', 'Coalesced', 'Tiled', 'CuBLAS']

colores_lin  = ['#8d94a1', '#5b7fa6', '#6a9e7f', '#9e7f6a', NEON]

x = np.arange(len(tamanos))

fig, ax = plt.subplots(figsize=(12, 7))

for idx, modo in enumerate(modos):
    valores = []
    for tam in tamanos:
        val = gflops_totales.get(f"{tam}_{modo}.csv", 1e-5)
        valores.append(max(val, 1e-5))

    color = colores_lin[idx]
    ax.plot(x, valores, label=etiquetas[idx], color=color, linewidth=2.5,
            marker='o', markersize=9, markerfacecolor=color,
            markeredgecolor=FG, markeredgewidth=1.5)
    ax.text(x[-1] + 0.1, valores[-1], etiquetas[idx], color=color, fontsize=12, va='center')

ax.set_yscale('log')

ax.set_xticks(x)
ax.set_xticklabels([f"EXP {t}" for t in tamanos], fontsize=14)
ax.tick_params(axis='x', direction='inout', length=8, width=1.2)
ax.tick_params(axis='y', labelsize=14)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.2)
ax.spines['bottom'].set_linewidth(1.2)
ax.spines['left'].set_color(FG)
ax.spines['bottom'].set_color(FG)

plt.subplots_adjust(right=0.85)
ax.set_title("Crecimiento de GFLOPS", fontsize=22, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 7. GFLOPS para EXP 14

In [ ]:
exponente     = '14'
tamano_matriz = 2 ** int(exponente)
modos         = ['cpu', 'gpu_naive', 'gpu_coalesced', 'gpu_tiled', 'gpu_cublas']
etiquetas_14  = ['CPU', 'Naive', 'Coalesced', 'Tiled', 'CuBLAS']

valores = [gflops_totales.get(f"{exponente}_{m}.csv", 0) for m in modos]
x       = np.arange(len(modos))
max_val = max(valores) if max(valores) > 0 else 1

fig, ax = plt.subplots(figsize=(11, 6.5))

# Barras con degradado
for i, (xpos, val) in enumerate(zip(x, valores)):
    color_top = NEON if i == 4 else '#b0b8c8'
    grad = np.linspace(0.3, 1.0, 256).reshape(256, 1)
    ax.imshow(grad, extent=[xpos - 0.3, xpos + 0.3, 0, val],
              aspect='auto', origin='lower',
              cmap=mcolors.LinearSegmentedColormap.from_list("g", [BG, color_top]),
              zorder=2)


# Etiquetas encima de cada barra
for i, val in enumerate(valores):
    color_txt = NEON if i == 4 else FG
    ax.text(x[i], val + max_val * 0.03, f"{val:.0f}",
            color=color_txt, ha='center', va='bottom', fontsize=16, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=BG, alpha=0.6, edgecolor='none'))

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_14, fontsize=18, fontweight='bold')
ax.tick_params(axis='x', length=0, pad=15)
ax.set_yticks([])
ax.set_ylim(0, max_val * 1.25)
ax.set_xlim(-0.6, len(modos) - 0.4)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.spines['top'].set_visible(True)
ax.spines['top'].set_color(FG)
ax.spines['top'].set_linewidth(1.5)

ax.set_title(f"GFLOPS — EXP {exponente}", fontsize=22, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 8. Crecimiento del Tamaño de Matriz vs. Capacidad de Memoria

In [ ]:
exps = np.arange(10, 21)
sizes_bytes = (2 ** (exps * 2)) * 4  # float32

etiquetas_mem = ["4 MB", "16 MB", "64 MB", "256 MB",
                  "1 GB", "4 GB", "16 GB", "64 GB", "256 GB", "1 TB", "4 TB"]

MB = float(1024 ** 2)
GB = float(1024 ** 3)
TB = float(1024 ** 4)
limite_vram = 8.0  * GB
limite_ram  = 64.0 * GB
limite_ssd  = 512.0 * GB

color_vram_bg = '#1a2e1a';  color_vram_txt = '#ccff00'
color_ram_bg  = '#2a1e1e';  color_ram_txt  = '#ff6666'
color_ssd_bg  = '#1e1a2e';  color_ssd_txt  = '#aa88ff'
color_out_bg  = '#1a2230';  color_out_txt  = '#66aaff'

lim_izq = sizes_bytes[0]
lim_der = sizes_bytes[-1]

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(lim_izq, lim_der)

ax.axvspan(lim_izq,     limite_vram, color=color_vram_bg, alpha=1.0, zorder=1)
ax.axvspan(limite_vram, limite_ram,  color=color_ram_bg,  alpha=1.0, zorder=1)
ax.axvspan(limite_ram,  limite_ssd,  color=color_ssd_bg,  alpha=1.0, zorder=1)
ax.axvspan(limite_ssd,  lim_der,     color=color_out_bg,  alpha=1.0, zorder=1)

y_lbl = 20.6
ax.text(np.sqrt(lim_izq     * limite_vram), y_lbl, "Cabe en VRAM", color=color_vram_txt, ha='center', fontweight='bold', zorder=5)
ax.text(np.sqrt(limite_vram * limite_ram),  y_lbl, "Cabe en RAM",  color=color_ram_txt,  ha='center', fontweight='bold', zorder=5)
ax.text(np.sqrt(limite_ram  * limite_ssd),  y_lbl, "Cabe en SSD",  color=color_ssd_txt,  ha='center', fontweight='bold', zorder=5)
ax.text(np.sqrt(limite_ssd  * lim_der),     y_lbl, "Supera un PC", color=color_out_txt,  ha='center', fontweight='bold', zorder=5)

ax.plot(sizes_bytes, exps, color=NEON, linewidth=2.5, zorder=3)
ax.scatter(sizes_bytes, exps, color=NEON, s=80, edgecolor=BG, linewidth=1.5, zorder=4)
for sx, sy, lbl in zip(sizes_bytes, exps, etiquetas_mem):
    ax.text(sx * 0.8, sy - 0.5, lbl, va='center', fontsize=10, zorder=5)

ax.set_xscale('log', base=2)
ax.set_yticks(exps)
ax.set_yticklabels([f"EXP {e}" for e in exps], fontsize=11)
ax.set_ylim(9.5, 21)

# Ocultar completamente el eje X inferior
ax.set_xticks([])
ax.spines['bottom'].set_visible(False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(GRAY_BAR)

ax.set_xlabel(r"Tamaño de $A$", fontsize=13, labelpad=15)
plt.title(r"Crecimiento del tamaño de $A$ vs. capacidad de memoria",fontsize=16, fontweight='bold', pad=30)

plt.tight_layout()
plt.show()